In [4]:
from __future__ import annotations
import pandas as pd
from pathlib import Path
from topologicpy.PyG import PyG

renderer = "vscode"

PREDICTION_LEVEL = "graph"
TASK             = "classification"
GRAPH_LABEL_TYPE = "categorical"
NODE_LABEL_TYPE  = "categorical"
EDGE_LABEL_TYPE  = "categorical"

MODEL_PATH = Path(r"C:/Users/etmaglari/IAAC/etmaglari_gML/Homework03/CSV/pyg_model.pt").resolve()


# ----- PHASE 2: PREDICTION OF UNSEEN DATASET ------

## 1. Load Testing Dataset

In [7]:
#dataset_dir = Path(r"C:\Users\etmaglari\IAAC\etmaglari_gML\example_dataset\dataset_graph_classification").resolve()  # full dataset → all 5 labels
dataset_dir = Path(r"C:\Users\etmaglari\IAAC\etmaglari_gML\Homework03\Supporting File 2\CSV").resolve()  # your single building

mapping = {0: "Separation",
           1: "Separation with Plinth",
           2: "Adherence",
           3: "Adherence with Plinth",
           4: "Interlock"}

pyg_2 = PyG.ByCSVPath(
    path=str(dataset_dir),
    level=PREDICTION_LEVEL,
    task=TASK,
    graphLabelType=GRAPH_LABEL_TYPE,
    nodeLabelType=NODE_LABEL_TYPE,
    edgeLabelType=EDGE_LABEL_TYPE,
)

## 2. Load the Pre-trained Model

In [8]:
pyg_2.LoadModel(str(MODEL_PATH))

## 3. Make the Whole Dataset a Testing Dataset

In [9]:
pyg_2.SetHyperparameters(split=(0.0, 0.0, 1.0), shuffle=False)  # all graphs become test
print("PyG config summary:")
print(pyg_2.Summary())

PyG config summary:
{'level': 'graph', 'task': 'classification', 'graph_label_type': 'categorical', 'node_label_type': 'categorical', 'edge_label_type': 'categorical', 'cv': 'holdout', 'split': (0.0, 0.0, 1.0), 'k_folds': 5, 'holdout_group_by': None, 'conv': 'sage', 'hidden_dims': (128, 128), 'activation': 'relu', 'dropout': 0.2, 'batch_norm': True, 'residual': True, 'pooling': 'mean', 'epochs': 50, 'batch_size': 32, 'lr': 0.001, 'weight_decay': 0.0, 'optimizer': 'adam', 'gradient_clip_norm': None, 'early_stopping': False, 'early_stopping_patience': 10, 'device': 'cuda:0', 'ontology': True, 'ontology_metadata_columns': {'graph': ['label', 'graph_id'], 'node': ['label', 'graph_id', 'node_id'], 'edge': ['label', 'graph_id', 'src_id', 'dst_id']}, 'num_graphs': 1, 'num_outputs': 5}


## 4. Predict the Dataset

In [11]:
pred_results = pyg_2.Predict()
indices = pred_results['index'].tolist()
predictions = pred_results['pred'].tolist()
probabilities = pred_results['prob'].tolist()


prediction_labels = [mapping[prediction] for prediction in predictions]
df = pd.DataFrame({
    "index": indices,
    "prediction": predictions,
    "Label": prediction_labels,
    "confidence": [round(max(p), 2) for p in probabilities]
})

df

,index,prediction,Label,confidence
0,0,1,Separation with Plinth,1.0


## 5. Plot the Confusion Matrix (For Categorical Labels Only)

In [12]:
import numpy as np
from sklearn.metrics import confusion_matrix
from topologicpy.Plotly import Plotly
actual = np.array(pred_results["y_true"]).reshape(-1)
predicted = np.array(pred_results["pred"]).reshape(-1)
cm = confusion_matrix(actual, predicted, labels=[0,1,2,3,4])
fig = Plotly.FigureByConfusionMatrix(cm)
Plotly.Show(fig)